# Matchups of in situ data with satellite data

**Demo Leads:** James Allen (NASA, MSU), Anna Windle (NASA, SSAI)

## Summary

In this example we will conduct matchups of in situ AERONET-OC Rrs data with PACE OCI Rrs data. 

The Aerosol Robotic Network (AERONET) was developed to sustain atmospheric studies at various scales with measurements from worldwide distributed autonomous sun-photometers. This has been extended to support marine applications, called AERONET – Ocean Color [(AERONET-OC)](https://aeronet.gsfc.nasa.gov/new_web/ocean_levels_versions.html), and provides the additional capability of measuring the radiance emerging from the sea (i.e., water-leaving radiance) with modified sun-photometers installed on offshore platforms like lighthouses, oceanographic and oil towers. AERONET-OC is instrumental in satellite ocean color validation activities.

In this tutorial, we will be collecting Rrs data from the [AAOT](https://aeronet.gsfc.nasa.gov/cgi-bin/data_display_seaprism_v3?site=AAOT&nachal=2&level=2&place_code=10) AERONET-OC site located at 45.3N, 12.5E off the coast of Venice, Italy.

We will extract PACE OCI Rrs data in a 5x5 pixel window around the AERONET-OC site and compare it to in situ AERONET-OC Rrs observations.

## Learning Objectives

At the end of this notebook you will be able to:

- Access and parse data from a specific AERONET-OC site and date range
- Access and filter PACE OCI Rrs data from a specific region and date range
- Match satellite and in situ data spatially and temporally
- Visualize matchup performance with statistical plots (Bland-Altman and scatter)

Throughout, you'll gain experience with Python tools including `pandas`, `xarray`, and `matplotlib`, as well as real-world remote sensing workflows.

## Contents

1. [Setup](#1.-Setup)
2. [Process AERONET-OC data](#2.-Process-AERONET-OC-data)
3. [Download PACE OCI granules](#3.-Download-PACE-OCI-granules)
4. [Apply matchup code](#4.-Apply-matchup-code)
5. [Make plots](#5.-Make-plots)

## 1. Setup

We begin by loading a set of utility functions that work behind the scenes to do the majority of the work for us.

**Collapse this cell and just run it** — but feel free to explore it later to see how these functions are implemented. Understanding what's under the hood will deepen your skills!

<div class="alert alert-block alert-warning">
Note: The `get_f0` function requires an OCI sensor data file from the OCSSW shared files. Make sure your path is set up to be able to read the OCSSWROOT environment variable! If not, you can always download the F0 (TSIS-1 0.1nm) file from https://oceancolor.gsfc.nasa.gov/resources/docs/rsr_tables/
</div>

In [ ]:
"""Helper functions for PACE Hackweek Validation Tutorial.

Authors:
    James Allen and Anna Windle
"""

import datetime
import os
import re
from pathlib import Path

import earthaccess
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.style as style
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.ticker import FuncFormatter
from scipy import odr, stats
import requests

# AERONET-OC Download Constants
# Valid AERONET-OC site list
DF_AERONET_SITES = pd.read_csv(
    "https://aeronet.gsfc.nasa.gov/aeronet_locations_v3.txt",
    delimiter=",",
    skiprows=1
    )
AERONET_SITES = list(DF_AERONET_SITES["Site_Name"].sort_values())
OCEAN_SITES = [
    "AAOT",
    "Abu_Al_Bukhoosh",
    "ARIAKE_TOWER",
    "Bahia_Blanca",
    "Banana_River",
    "Blyth_NOAH",
    "Casablanca_Platform",
    "Chesapeake_Bay",
    "COVE_SEAPRISM",
    "Galata_Platform",
    "Gloria",
    "GOT_Seaprism",
    "Grizzly_Bay",
    "Gustav_Dalen_Tower",
    "Helsinki_Lighthouse",
    "Ieodo_Station",
    "Irbe_Lighthouse",
    "Kemigawa_Offshore",
    "Lake_Erie",
    "Lake_Okeechobee",
    "Lake_Okeechobee_N",
    "LISCO",
    "Lucinda",
    "MVCO",
    "Palgrunden",
    "PLOCAN_Tower",
    "RdP-EsNM",
    "Sacramento_River",
    "San_Marco_Platform",
    "Section-7_Platform",
    "Socheongcho",
    "South_Greenbay",
    "Thornton_C-power",
    "USC_SEAPRISM",
    "Venise",
    "WaveCIS_Site_CSI_6",
    "Zeebrugge-MOW1",
]

# Get subset of AERONET columns to make it a bit more manageable (also rename)
AOC_KEEP_COLS = [
    "AERONET_Site",
    "field_datetime",
    "Site_Latitude(Degrees)",
    "Site_Longitude(Degrees)",
    "Solar_Zenith_Angle[400nm]",
]
COLUMN_RENAME = {
    "Site_Latitude(Degrees)": "field_latitude",
    "Site_Longitude(Degrees)": "field_longitude",
    "AERONET_Site": "field_site",
    "Solar_Zenith_Angle[400nm]": "field_solar_zenith",
}

# Bland-Altman/Scatterplot Constants
# Plot colors, font sizes
COLOR_PALETTE = sns.color_palette("colorblind")
COLOR_SCATTER = COLOR_PALETTE[0]
COLOR_LINE = "black"  # Was "black"
COLOR_LOA = COLOR_PALETTE[2]  # Was "green"
COLOR_FITLINE = COLOR_PALETTE[1]  # Was "magenta"
SIZE_TITLE = 24
SIZE_AXLABEL = 20
SIZE_TEXTLABEL = 14
SHOW_LEGEND = False

# Update some defaults
plt.rcParams.update({"figure.dpi": 300})
sns.set_style("ticks", rc={"figure.dpi": 300})
sns.set_context("notebook", font_scale=1.45)

# Satellite Matchup Constants
# Short names for earthaccess lookup
SAT_LOOKUP = {
    "PACE_AOP": "PACE_OCI_L2_AOP",
    "PACE_IOP": "PACE_OCI_L2_IOP",
    "PACE_BGC": "PACE_OCI_L2_BGC",
    "PACE_PAR": "PACE_OCI_L2_PAR",
    "AQUA": "MODISA_L2_OC",
    "TERRA": "MODIST_L2_OC",
    "NOAA-20": "VIIRSJ1_L2_OC",
    "NOAA-21": "VIIRSJ2_L2_OC",
    "SUOMI-NPP": "VIIRSN_L2_OC",
}

# List l2 flags, then build them into a dict
l2_flags_list = [
    "ATMFAIL",
    "LAND",
    "PRODWARN",
    "HIGLINT",
    "HILT",
    "HISATZEN",
    "COASTZ",
    "SPARE",
    "STRAYLIGHT",
    "CLDICE",
    "COCCOLITH",
    "TURBIDW",
    "HISOLZEN",
    "SPARE",
    "LOWLW",
    "CHLFAIL",
    "NAVWARN",
    "ABSAER",
    "SPARE",
    "MAXAERITER",
    "MODGLINT",
    "CHLWARN",
    "ATMWARN",
    "SPARE",
    "SEAICE",
    "NAVFAIL",
    "FILTER",
    "SPARE",
    "BOWTIEDEL",
    "HIPOL",
    "PRODFAIL",
    "SPARE",
]
L2_FLAGS = {flag: 1 << idx for idx, flag in enumerate(l2_flags_list)}

# Bailey and Werdell 2006 exclusion criteria
EXCLUSION_FLAGS = [
    "LAND",
    "HIGLINT",
    "HILT",
    "STRAYLIGHT",
    "CLDICE",
    "ATMFAIL",
    "LOWLW",
    "FILTER",
    "NAVFAIL",
    "NAVWARN",
]

# OCSSW Dataroot folder for tables
# OCDATAROOT = Path(os.environ.get("OCSSWROOT")).resolve() / "share"
OCDATAROOT = "/private/tmp/ocssw/share/"
OCI_SENSOR_FILE = OCDATAROOT + "oci/msl12_sensor_info.dat"

##---------------------------------------------------------------------------##
#                              General Utilities                              #
##---------------------------------------------------------------------------##


def get_f0(wavelengths=None, window_size=10):
    """Load the OCI sensor file and return F0.

    Defaults to returning the full table. Input obs_time to correct for the
    Earth-Sun distance.

    Parameters
    ----------
    sensor_file : str or pathlib.Path
        Path to the OCI satellite sensor file containing wavelengths and F0.
    wavelengths : array-like, optional
        Wavelengths at which to compute the average irradiance.
        If None, returns the full wavelength and irradiance table.
    window_size : int, optional
        Bandpass filter size for mean filtering to selected wavelengths, in nm.

    Returns
    -------
    tuple of np.ndarray
        A tuple containing:
        - f0_spectra : np.ndarray
            The extraterrestrial solar irradiance, in uW/cm^2/nm.
        - f0_wave : np.ndarray
            The corresponding wavelengths, in nm.

    """
    with open(OCI_SENSOR_FILE, "r") as file_in:
        for line in file_in:
            if "Nbands" in line:
                (key, nbands) = line.split("=")
                break

    wl = np.zeros(int(nbands), dtype=float)
    f0 = np.zeros(int(nbands), dtype=float)
    with open(OCI_SENSOR_FILE, "r") as file_in:
        for line in file_in:
            if "=" in line:
                (key, value) = line.split("=")
                if "Lambda" in key:
                    idx = re.findall(r"\d+", key)
                    wvlidx = int(idx[0]) - 1
                    wl[wvlidx] = float(value)
                if "F0" in key:
                    idx = re.findall(r"\d+", key)
                    wvlidx = int(idx[1]) - 1
                    f0[wvlidx] = float(value)

    if wavelengths is not None:
        f0_wave = np.array(wavelengths)
        f0_spectra = bandpass_avg(f0, wl, window_size, f0_wave)
    else:
        f0_wave = wl
        f0_spectra = f0

    return f0_spectra, f0_wave


def bandpass_avg(
        data,
        input_wavelengths,
        window_size=10,
        target_wavelengths=None
        ):
    """Apply a band-pass filter to the data.

    Parameters
    ----------
    data : np.ndarray
        1D or 2D array containing the spectral data (samples x wavelengths).
        If 1D, it's assumed to be a single sample.
    input_wavelengths : np.ndarray
        1D array of wavelength values corresponding to the columns of data.
    window_size : int, optional
        Size of the window to use for averaging. Default is 10 nm.
    target_wavelengths : np.ndarray, optional
        1D array of target wavelengths for filtered values.
        If None, the input wavelengths are used.

    Returns
    -------
    np.ndarray
        1D or 2D array containing the band-pass filtered data.

    """
    data = np.atleast_2d(data)
    half_window = window_size / 2
    num_samples, num_input_wavelengths = data.shape
    if target_wavelengths is None:
        target_wavelengths = input_wavelengths

    filtered_data = np.empty((num_samples, len(target_wavelengths))) * np.nan

    for idx, target_wl in enumerate(target_wavelengths):
        start = target_wl - half_window
        end = target_wl + half_window
        cols_in_range = np.where(
            (input_wavelengths >= start) & (input_wavelengths <= end)
        )[0]
        if cols_in_range.size > 0:
            filtered_data[:, idx] = np.nanmean(data[:, cols_in_range], axis=1)

    return filtered_data if num_samples > 1 else filtered_data.flatten()


def get_column_prods(df, type_prefix):
    """Process a dataframe to create a dictionary of data products.

    Parameters
    ----------
    df : pandas DataFrame
        Extracted dataframes from read_extract_file
    type_prefix : str
        Prefix to identify the product columns, e.g. "aoc"

    Returns
    -------
    data_dict
        dictionary mapping data product with their wavelengths and columns.

    """
    data_dict = {}
    pattern = rf"{type_prefix}_(\w+?)(\d*\.?\d+)?$"

    for col in df.columns:
        match = re.match(pattern, col)
        if match:
            product = match.group(1)
            wavelength = match.group(2) if match.group(2) else None
            if product not in data_dict:
                data_dict[product] = {"wavelengths": [], "columns": []}
            data_dict[product]["columns"].append(col)
            if wavelength:
                if "." in wavelength:
                    data_dict[product]["wavelengths"].append(float(wavelength))
                else:
                    data_dict[product]["wavelengths"].append(int(wavelength))
    return data_dict


def read_sb(filename_sb):
    """Read SeaBASS file and returns just the data.

    Input
    -----
    filename_sb : str
        path to seabass file

    Output
    ------
    data : pandas dataframe object
        seabass data from file
    """
    with open(filename_sb, "r") as file:
        lines = [line.rstrip() for line in file]

    # Parse headers, get index where they end
    idx_endheader = [index for index, value in enumerate(lines)
                     if value == "/end_header"]
    header_lines = lines[1:idx_endheader[0]]
    headers = dict()
    comments = []
    for header_line in header_lines:
        if header_line.startswith("!"):
            # Separate out the comments
            comments.append(header_line)
        else:
            # Split the header and add to the dictionary
            key, value = header_line.split("=", 1)
            headers[key[1:]] = value  # Remove leading "/" from key

    # Pull data into pandas dataframe
    data = pd.read_csv(filename_sb,
                       skiprows=idx_endheader[0]+1,
                       names=headers["fields"].split(","),
                       na_values=headers["missing"])

    # Index by datetime
    get_sb_datetime(data)

    return data


def get_sb_datetime(df):
    """Parse datetime from different combinations of dates and times."""
    if all(col in df.columns for col in ["year", "month", "day",
                                         "hour", "minute", "second"]):
        df["datetime"] = pd.to_datetime(df[["year", "month", "day",
                                            "hour", "minute", "second"]])
    elif all(col in df.columns for col in ["year", "month", "day", "time"]):
        df["datetime"] = pd.to_datetime(
            df["year"].astype(str) + df["month"].astype(str).str.zfill(2)
            + df["day"].astype(str).str.zfill(2) + ' ' + df["time"])
    elif all(col in df.columns for col in ["date", "time"]):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + ' ' + df["time"])
    elif all(col in df.columns for col in ["year", "month", "day"]):
        df["datetime"] = pd.to_datetime(df[["year", "month", "day"]])
    elif all(col in df.columns for col in ["date", "hour",
                                           "minute", "second"]):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + ' ' + df["hour"].astype(str).str.zfill(2)
            + ':' + df["minute"].astype(str).str.zfill(2) + ':'
            + df["second"].astype(str).str.zfill(2))
    else:
        print("Unrecognized date/time format in DataFrame columns."
              "\nMay be a profile, but doublecheck.")
        return

    # Reindex the dataframe with the new datetime
    df.set_index("datetime", inplace=True)


##---------------------------------------------------------------------------##
#                             Satellite Utilities                             #
##---------------------------------------------------------------------------##


def parse_quality_flags(flag_value):
    """Parse bitwise flag into a list of flag names.

    Parameters
    ----------
    flag_value : int
        The integer representing the combined bitwise quality flags.

    Returns
    -------
    list of str
        List of flag names that are set in the flag_value.

    """
    return [
        flag_name for flag_name, value in L2_FLAGS.items()
        if (flag_value & value) != 0
    ]


def get_fivebyfive_Rrs(file, latitude, longitude, par, wavelengths, rrs_wavelengths):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups
    wavelengths ; numpy array
        Desired Rrs wavelengths to match
    rrs_wavelengths ; numpy array
        Rrs wavelengths (from wavelength_3d for OCI)

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: This is hard-coded to Rrs from an L2 AOP file.
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        rrs_data = (
            ds_data[par].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        
        # Select only the desired wavelengths
        rrs_indices = [np.where(rrs_wavelengths == wl)[0][0] for wl in wavelengths]
        rrs_data = rrs_data[:, :, rrs_indices]

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any():
        rrs_valid = rrs_data[valid_mask]
        rrs_std_initial = np.std(rrs_valid, axis=0)
        rrs_mean_initial = np.mean(rrs_valid, axis=0)

        # Exclude spectra > 1.5 stdevs away
        std_mask = np.all(
            np.abs(rrs_valid - rrs_mean_initial) <= 1.5 * rrs_std_initial,
            axis=1
        )
        rrs_std = np.std(rrs_valid[std_mask], axis=0)
        rrs_mean = np.mean(rrs_valid[std_mask], axis=0).flatten()
    else:
        rrs_mean = np.nan * np.empty_like(wavelengths)

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
    }

    # Add mean spectra to the row dictionary
    for wavelength, mean_value in zip(wavelengths, rrs_mean):
        key = f"sat_{par.lower()}{int(wavelength)}"
        row[key] = mean_value

    return row


def get_fivebyfive_PAR(file, latitude, longitude):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: Using "par_day_planar_above" product for PAR data, which I assume is what standard OC PAR is...
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        par_data = (
            ds_data["par_day_planar_above"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any():
        par_valid = par_data[valid_mask]
        par_mean = np.mean(par_valid, axis=0).flatten()
    else:
        par_mean = np.nan

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
        "sat_par": par_mean
    }

    return row


def get_sat_matchups(
    start_date,
    end_date,
    latitude,
    longitude,
    wavelengths="all",
    par="Rrs",
    sat="PACE_AOP",
    selected_dates=None
):
    """Make satellite timeseries of matchups from single station.

    Caution: If the date or coordinates aren't formatted correctly, it might
    pull a huge granule list and take forever to run. If it takes more than 45
    seconds to print the number of granules, just kill the process.

    Uses the earthaccess package. Defaults to the PACE OCI L2 IOP datasets,
    but other satellites can be used if they have a corresponding short_name
    in the SAT_LOOKUP dictionary.

    Workflow:
        1. Get list of matchup granules
        2. Loop through each file and:
            2a. Find closest pixel to station, extract 5x5 pixel box
            2b. Exclude pixels based on l2_flags
            2c. Filtered mean to get single spectra
            2d. Compute statistics and save data row
        3. Organize output pandas dataframe

    Parameters
    ----------
    start_date : datetime or str
        Beginning of Aeronet data to run.
    end_date : datetime or str, optional
        End of Aeronet data to run.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups
    sat : str
        Name of satellite to search. Must be in SAT_LOOKUP dict constant.
    selected_dates : list of str, optional
        If given, only pull granules if the dates are in this list

    Returns
    -------
    pandas DataFrame object
        Flattened table of all satellite granule matchups.

    """
    # Look up short name from constants
    if sat not in SAT_LOOKUP.keys():
        raise ValueError(
            f"{sat} is not in the lookup dictionary. Available "
            f"sats are: {', '.join(SAT_LOOKUP)}"
        )
    short_name = SAT_LOOKUP[sat]

    # Format search parameters
    time_bounds = (f"{start_date}T00:00:00", f"{end_date}T23:59:59")

    # Run Earthaccess data search
    results = earthaccess.search_data(
        point=(longitude, latitude),
        temporal=time_bounds,
        short_name=short_name
    )
    if selected_dates is not None:
        filtered_results = [
            result
            for result in results
            if result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10]
            in selected_dates
        ]
        print(f"Filtered to {len(filtered_results)} Granules.")
        files = earthaccess.open(filtered_results)
    else:
        files = earthaccess.open(results)

    # Get 5x5 pixel data
    if sat == "PACE_AOP" or sat == "PACE_IOP":
        # Pull out all available Rrs wavelengths
        with xr.open_dataset(files[0], group="sensor_band_parameters") as ds_bands:
            rrs_wavelengths = ds_bands["wavelength_3d"].values
        
        # Select wavelengths or interest
        if wavelengths is None or wavelengths == "all":
            wavelengths = rrs_wavelengths
        else:
            # Get nearest wavelengths to desired input wavelengths
            nearest_wavelengths = []
            for target_wl in wavelengths:
                nearest_wl = rrs_wavelengths[np.abs(rrs_wavelengths - target_wl).argmin()]
                nearest_wavelengths.append(nearest_wl)
            wavelengths = np.array(nearest_wavelengths)
        
        # Loop through files and process
        sat_rows = []
        for idx, file in enumerate(files):
            granule_date = pd.to_datetime(
                file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
            )
            print(f"Running Granule: {granule_date}")
            row = get_fivebyfive_Rrs(file, latitude, longitude, par, wavelengths, rrs_wavelengths)
            sat_rows.append(row)
            
    elif sat == "PACE_PAR":
        # Loop through files and process
        sat_rows = []
        for idx, file in enumerate(files):
            granule_date = pd.to_datetime(
                file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
            )
            print(f"Running Granule: {granule_date}")
            row = get_fivebyfive_PAR(file, latitude, longitude)
            sat_rows.append(row)
        
    return pd.DataFrame(sat_rows)
    

For those starting this demonstration fresh, we'll double-check that we've got our earthaccess credentials set up!

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True)

## 2. Process AERONET-OC data

We will use the function `process_aeronet` to download and process AERONET-OC data from the 'AAOT' site. We will filter Level 1.5 data from the dates June 1, 2024 to July 31, 2024. This function will output a pandas dataframe of every AERONET-OC record between the dates.

There are three "levels" of AERONET-OC data in terms of data quality: 1, 1.5, and 2. If a complete measurement sequence with the instruments is able to be performed, it is collected and stored as Level 1. These data are then passed through an automated quality control system and stored as Level 1.5 if they pass all tests. Finally, Level 2 data are data from Level 1.5 that are subsequently screened by an experienced scientist and validated. We'll be using Level 1.5 data to pull as much good quality data as possible without the time lag for manual validation. More information on AERONET-OC levels can be found in [Zibordi et al., 2009.](https://doi.org/10.1175/2009JTECHO654.1)

In [ ]:
df_aeronet = process_aeronet(
    aeronet_site="AAOT",
    start_date="2024-03-01",
    end_date="2025-06-30",
    data_level=15,
)
df_aeronet.head()

## 3. Download PACE OCI granules

We will use the function `get_sat_ts_matchups` to search for `PACE_OCI_L2_AOP_NRT` data using `earthaccess` within the specified time range and at the (lat,lon) coordinate of the AAOT AERONET-OC site. This function finds the closest pixel and extracts all data within a 5x5 pixel window, excludes pixels based on L2 flags, calculates the mean to retrive a single Rrs spectra, and computes matchup statistics. The function outputs a pandas dataframe of every `PACE_OCI_L2_AOP` Rrs spectra for the specified time range. We'll also include an optional list of unique date strings from the AERONET-OC dataframe to "skip" the granules that don't have any field data associated with them.

<div class="alert alert-block alert-warning">
Note: This section will actually take quite a while to pull enough granules for the plotting section to give us robust stats (at least 35 valid matchups are needed), so we'll be skipping this part in favor of a pre-made dataset for the demo.
</div>

In [ ]:
tag_df = pd.read_csv(
    "tag_dataframe.csv",
    delimiter=",",
    skiprows=0
)
tag_df["all_dates"] = pd.to_datetime(tag_df["all_dates"])

tag_ids = tag_df["all_tagID"].values
tag_profilnum = tag_df["all_profn"].values
tag_lat = tag_df["all_lats"].values
tag_lon = tag_df["all_lons"].values
tag_dates = tag_df["all_dates"].dt.strftime("%Y-%m-%d").tolist()

# Loop through each tag position and date
go_to_next_day = False
for i, (lat, lon, date, tag, profile) in enumerate(zip(tag_lat, tag_lon, tag_dates, tag_ids, tag_profilnum)):
    if i < 5:  # Skip to index 5 ()
        continue
    
    print(f"Processing tag {tag}, profile {profile}/{len(tag_lat)}: lat={lat}, lon={lon}, date={date}")

    if go_to_next_day:
        if date == date_to_skip:
            print(f"Skipping date {date} as previously determined.")
            continue
        else:
            go_to_next_day = False
    
    # Extract Rrs from PACE AOP
    df_Rrs = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        par="Rrs",
        sat="PACE_AOP",
        selected_dates=[date],
        )
    
    # If valid Rrs data found, extract Kd from PACE IOP
    if df_Rrs["sat_pixel_valid"].sum()>0:
        df_Kd = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            wavelengths=[490],
            par="Kd",
            sat="PACE_IOP",
            selected_dates=[date],
            )
        
        df_PAR = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            par="PAR",
            sat="PACE_PAR",
            selected_dates=[date],
            )
        
        # Merge Rrs, Kd, and PAR dataframes on sat_datetime
        df_satellite = pd.merge(
            df_Rrs,
            df_PAR,
            on=["sat_datetime", "sat_latitude", "sat_longitude", "sat_pixel_valid"],
            how="outer"
        )
        df_satellite = pd.merge(
            df_satellite,
            df_Kd,
            on=["sat_datetime", "sat_latitude", "sat_longitude", "sat_pixel_valid"],
            how="outer"
        )
        
        # TESTING: Print out the tag and profile number for tracking and break
        print(tag, profile)
        break
    
    # If no valid data, skip to next day
    if df_Rrs["sat_pixel_valid"].sum()==0:
        print(f"No valid satellite data for date {date}.")
        date_to_skip = date
        go_to_next_day = True


In [ ]:
print(df_satellite)

# Save to CSV
df_satellite.to_csv(f"satellite_matchups_tag{tag}_prof{profile}.csv", index=False)

In [ ]:
# Pull out coordinates
aeronet_lat = df_aeronet["field_latitude"][0]
aeronet_lon = df_aeronet["field_longitude"][0]

# Pull out unique days
unique_days = df_aeronet["field_datetime"].dt.date.unique()
unique_days_str = [day.strftime("%Y-%m-%d") for day in unique_days]

# df_satellite = get_sat_ts_matchups(
#     start_date="2024-03-01",
#     end_date="2024-03-05",
#     latitude=aeronet_lat,
#     longitude=aeronet_lon,
#     sat="PACE_AOP",
#     selected_dates=unique_days_str,
# )

df_satellite = get_sat_ts_matchups(
    start_date="2025-01-30",
    end_date="2025-01-30",
    latitude=-54.7682,
    longitude=77.8958,
    sat="PACE_AOP",
    selected_dates='2025-01-30',
)

In [ ]:
print(df_satellite)

## 3b. Precooked Download PACE OCI granules

Since the previous section takes quite a while to pull the granules we need, here's a premade data pull of PACE OCI data (for AAOT).

In [ ]:
df_satellite = pd.read_csv("/home/jovyan/shared-public/pace-hackweek/oci_quickpull_aaot.csv")
df_satellite["sat_datetime"] = pd.to_datetime(df_satellite["sat_datetime"], utc=True)

In [ ]:
df_satellite.head()

## 4. Apply matchup code

We will use the function `match_data` to create a matchup dataframe based on selection criteria. This function defaults to using the [Bailey and Werdell 2006](https://oceancolor.gsfc.nasa.gov/staff/jeremy/bailey_and_werdell_2006_rse.pdf) matchup criteria, which reduces the measurements made at a given station to one representative sample for validating against the satellite spectra. Data are filtered based on the solar zenith angle, their noise level, and the time difference (here 180 minutes from the satellite overpass). Potential satellite matchups are also reduced based on the signal to noise level of the 5x5 pixel aggregation.

In [ ]:
?match_data

In [ ]:
matchups = match_data(
    df_satellite,
    df_aeronet,
    cv_max=0.60,
    senz_max=60.0,
    min_percent_valid=55.0,
    max_time_diff=180,
    std_max=1.5,
)
matchups

Pull out wavelengths and Rrs data from matchups

In [ ]:
dict_aoc = get_column_prods(matchups, "field")
waves_aoc = np.array(dict_aoc["rrs"]["wavelengths"])
rrs_aoc = matchups[dict_aoc["rrs"]["columns"]].to_numpy()

dict_sat = get_column_prods(matchups, "sat")
waves_sat = np.array(dict_sat["rrs"]["wavelengths"])
rrs_sat = matchups[dict_sat["rrs"]["columns"]].to_numpy()

## 4. Make plots

We will use the function `plot_BAvsScat` to plot the paired matchup data as Bland_Altman and scatter plots. The Bland-Altman plots provide insights into the bias and precision of the satellite measurements compared to field measurements. A mean difference close to zero indicates good agreement, while the spread of differences (limits of agreement) puts the bias within the context of the variability of the field data. Additionally, a check is done to assess the scale dependency of the bias, such as errors increasing when the magnitude of the observations increases. If a scale dependency exists, the limits of agreement are replaced with a regression line showing its direction and magnitude. Scatter plots complement Bland-Altman plots by showing the strength of the linear relationship between the two datasets, with high correlation coefficients and low RMSE values indicating strong agreement and high accuracy of the satellite-derived measurements.

In [ ]:
?plot_BAvsScat

In [ ]:
MATCH_WAVES = np.array([400, 412, 443, 490, 510, 560, 620, 667])

# Loop through matchup wavelengths
stats_list = []
for idx, match_wave in enumerate(MATCH_WAVES):
    # Find matching OCI columns
    idx_sat = np.where(np.abs(waves_sat - match_wave) <= 5)[0]
    match_sat = np.nanmean(rrs_sat[:, idx_sat], axis=1)

    # Find matching AOC columns
    idx_aoc = np.where(np.abs(waves_aoc - match_wave) <= 5)[0]
    match_aoc = np.nanmean(rrs_aoc[:, idx_aoc], axis=1)

    valid_indices = np.isfinite(match_sat) & np.isfinite(match_aoc)
    if np.sum(valid_indices) > 5:
        fig_label = f"Rrs({match_wave}), sr" + r"$\mathregular{^{-1}}$"
        dict_stats = plot_BAvsScat(
            match_aoc[valid_indices],
            match_sat[valid_indices],
            label=fig_label,
            saveplot=None,
            x_label="AeronetOC",
            y_label="PaceOCI",
            is_type2=True,
        )
        dict_stats["wavelength"] = match_wave
        stats_list.append(dict_stats)

# Organize stats DataFrame
df_stats = pd.DataFrame(stats_list)
df_stats.set_index("wavelength", inplace=True)
df_stats = df_stats.fillna(-999)
df_stats